In [1]:
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")

e:\Transformer_NLP\nlp_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
text="She drinks water"

tokens=tokenizer(text)

print(tokens)

{'input_ids': [101, 2016, 8974, 2300, 102], 'token_type_ids': [0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1]}


In [3]:
tokenizer.tokenize(text)

['she', 'drinks', 'water']

In [4]:
# let's modify littile bit of embeddings step
Tokens=tokenizer(text,return_tensors='pt',add_special_tokens=False)

# return_tensors='pt' : means return pytorch tensors insted of python lists
# add_special_tokens=False: This prevents adding [CLS],[SEP] so input becomes "She drinks water insted " of "[CLS] She drinks water [SEP]"

print(Tokens)
print(tokenizer.tokenize(text))

{'input_ids': tensor([[2016, 8974, 2300]]), 'token_type_ids': tensor([[0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1]])}
['she', 'drinks', 'water']


In [5]:
input_ids=Tokens['input_ids']
input_ids

tensor([[2016, 8974, 2300]])

In [6]:
import torch.nn as nn
vocab_size= tokenizer.vocab_size
hidden_size=8

embedding_layer=nn.Embedding(vocab_size,hidden_size)

In [7]:
embeddings=embedding_layer(input_ids)

print(f"Embeddings shape {embeddings.shape}")
print(embeddings)

Embeddings shape torch.Size([1, 3, 8])
tensor([[[-0.1301,  1.2443,  0.5716, -0.3841,  0.3377, -1.1910, -1.3888,
          -0.1659],
         [ 0.1750, -0.3266, -1.4852, -0.8066,  0.2271, -0.6807, -0.8606,
           1.5309],
         [-0.5640, -0.9320,  1.3163,  2.1454,  2.0053, -1.0256,  0.3462,
           1.5672]]], grad_fn=<EmbeddingBackward0>)


# Multihead attention 

## Step1 : 

In [15]:
num_heads=2
hidden_size=8
head_dim = hidden_size // num_heads

print(f"Head_dimension:{head_dim}")

Head_dimension:4


total hidden dimesion=8

split into 2 heads<br>
Each head gets 4 dimensions


## Step 2 Create Q,K,V same as single head attention model

[ d1 d2 d3 d4 d5 d6 d7 d8 ]

In [9]:
import torch.nn as nn
wq=nn.Linear(hidden_size,hidden_size)
wk=nn.Linear(hidden_size,hidden_size)
wv=nn.Linear(hidden_size,hidden_size)

In [10]:
Q=wq(embeddings)
K=wq(embeddings)
V=wq(embeddings)

In [11]:
print(f"Shape of Q: {Q.shape}")
print(f"Shape of K: {K.shape}")
print(f"Shape of V: {V.shape}")

Shape of Q: torch.Size([1, 3, 8])
Shape of K: torch.Size([1, 3, 8])
Shape of V: torch.Size([1, 3, 8])


### Step 3 : Split into 2 Heads 

In [16]:
batch_size, seq_len, _ = Q.shape

Q = Q.view(batch_size, seq_len, num_heads, head_dim)
K = K.view(batch_size, seq_len, num_heads, head_dim)
V = V.view(batch_size, seq_len, num_heads, head_dim)

print("After split Q shape:", Q.shape)

After split Q shape: torch.Size([1, 3, 2, 4])


Meaning:

1 batch

3 tokens

2 heads

4 dimensions per head

So:

Head 1 will compute attention using its 4 dimensions. <br>
Head 2 will compute attention using its 4 dimensions.

So now we have 

Token 1 → Head1, Head2<br>
Token 2 → Head1, Head2<br>
Token 3 → Head1, Head2

But we want <br>
Head 1 → Token1, Token2, Token3<br>
Head 2 → Token1, Token2, Token3

In [17]:
Q = Q.permute(0, 2, 1, 3)
K = K.permute(0, 2, 1, 3)
V = V.permute(0, 2, 1, 3)

print(Q.shape)

torch.Size([1, 2, 3, 4])


Meaning:

1 batch

2 heads

3 tokens

4 dimensions

## Step 4 : Now Compute Attention Score

In [18]:
import math
import torch

scores = torch.matmul(Q, K.transpose(-2, -1))

print("Scores shape:", scores.shape)

Scores shape: torch.Size([1, 2, 3, 3])


1 batch

2 heads

3 tokens

3 tokens

So now:

Head 1 has its own 3×3 attention matrix<br>
Head 2 has its own 3×3 attention matrix

For each head:

(
3
×
4
)
⋅
(
4
×
3
)
=
(
3
×
3
)
(3×4)⋅(4×3)=(3×3)

## Step 6 :Scale the Scores 

divide by root dk
 

In [19]:
scaled_scores = scores / math.sqrt(head_dim)

print("Scaled scores shape:", scaled_scores.shape)

Scaled scores shape: torch.Size([1, 2, 3, 3])


## Step7: Apply Softmax (Per Head)

In [20]:
attention_weights = torch.softmax(scaled_scores, dim=-1)

print("Attention weights shape:", attention_weights.shape)

Attention weights shape: torch.Size([1, 2, 3, 3])


In [21]:
print(attention_weights.sum(dim=-1))

tensor([[[1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000]]], grad_fn=<SumBackward1>)


## STEP 8:Multiply Attention Weights with V

In [22]:
Z = torch.matmul(attention_weights, V)

print("Z shape:", Z.shape)

Z shape: torch.Size([1, 2, 3, 4])


1 batch

2 heads

3 tokens

4 dimensions per head


So now you have:

Z1 = output of head 1<br>
Z2 = output of head 2

 ## STEP 9 : Rearrange Back Before Concatenation

In [23]:
Z = Z.permute(0, 2, 1, 3)

print("After permute:", Z.shape)

After permute: torch.Size([1, 3, 2, 4])


## STEP 10 : Concatenate Heads


In [24]:
Z = Z.contiguous().view(batch_size, seq_len, hidden_size)

print("After concatenation:", Z.shape)

After concatenation: torch.Size([1, 3, 8])


Head1_output (4 dims)<br>
+<br>
Head2_output (4 dims)<br>
=<br>
Combined_output (8 dims)

In [25]:
Wo = nn.Linear(hidden_size, hidden_size)

final_output = Wo(Z)

print("Final output shape:", final_output.shape)

Final output shape: torch.Size([1, 3, 8])


![alternative text](image\Multihead.png)